<a href="https://colab.research.google.com/github/PrajwalSathyanarayana/smollm2-medical-sft/blob/main/notebooks/01_data_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Clone GitHub Repo

In [1]:
# Clone the MedQUAD GitHub repository (raw XML source data)
# Using the original GitHub source (instead of a pre-cleaned HuggingFace version)
# gives full control over parsing and preserves all qtypes, including drug-related ones
!git clone https://github.com/abachaa/MedQuAD.git

fatal: destination path 'MedQuAD' already exists and is not an empty directory.


2. Inspect the folder structure

In [2]:
# List contents of the current working directory to confirm the repo was cloned
! ls

drive  MedQuAD	medquad_cleaned.csv  medquad_sft.jsonl	sample_data


In [3]:
# List the top-level folders inside MedQuAD
# Each folder corresponds to one NIH source (e.g., CancerGov, GARD, MPlusDrugs)
! ls MedQuAD

10_MPlus_ADAM_QA	     6_NINDS_QA
11_MPlusDrugs_QA	     7_SeniorHealth_QA
12_MPlusHerbsSupplements_QA  8_NHLBI_QA_XML
1_CancerGov_QA		     9_CDC_QA
2_GARD_QA		     LICENSE.txt
3_GHR_QA		     QA-TestSet-LiveQA-Med-Qrels-2479-Answers.zip
4_MPlus_Health_Topics_QA     readme.txt
5_NIDDK_QA


In [4]:
# Inspect files inside a single source folder to understand the file naming pattern
# Each .xml file represents one disease/drug document (may contain multiple Q&A pairs)
! ls MedQuAD/1_CancerGov_QA/


0000001_1.xml  0000006_3.xml	0000013_3_2.xml  0000024_2.xml	0000031_1.xml
0000001_2.xml  0000006_4.xml	0000013_3_3.xml  0000024_3.xml	0000031_2.xml
0000001_3.xml  0000006_5.xml	0000013_3_4.xml  0000024_4.xml	0000032_1.xml
0000001_4.xml  0000006_6.xml	0000013_3.xml	 0000024_5.xml	0000032_2.xml
0000001_5.xml  0000006_7.xml	0000014_1.xml	 0000024_6.xml	0000032_3.xml
0000001_6.xml  0000006_8.xml	0000014_2.xml	 0000024_7.xml	0000032_4.xml
0000001_7.xml  0000006_9.xml	0000014_3.xml	 0000024_8.xml	0000033_1.xml
0000003_1.xml  0000007_1.xml	0000014_4.xml	 0000024_9.xml	0000034_1.xml
0000003_2.xml  0000007_2.xml	0000015_1.xml	 0000025_1.xml	0000035_1.xml
0000003_3.xml  0000007_3.xml	0000016_1.xml	 0000025_2.xml	0000036_1.xml
0000003_4.xml  0000007_4.xml	0000017_1.xml	 0000026_1.xml	0000036_2.xml
0000003_5.xml  0000007_5.xml	0000018_1.xml	 0000026_2.xml	0000036_3.xml
0000003_6.xml  0000009_1.xml	0000019_1.xml	 0000026_3.xml	0000037_1.xml
0000004_1.xml  0000009_2.xml	0000019_2.xml	 0000027_1.xml	

In [5]:
# View the raw XML structure of a single document
# This reveals the schema: Document -> Focus -> QAPairs -> Question (qid, qtype) + Answer
! cat MedQuAD/1_CancerGov_QA/0000001_1.xml

<?xml version="1.0" encoding="UTF-8"?>
<Document id="0000001_1" source="CancerGov" url="https://www.cancer.gov/types/leukemia/patient/adult-all-treatment-pdq">
<Focus>Adult Acute Lymphoblastic Leukemia</Focus>
<FocusAnnotations>
	<UMLS>
		<CUIs>
			<CUI>C0751606</CUI>
		</CUIs>
		<SemanticTypes>
			<SemanticType>T191</SemanticType>
		</SemanticTypes>
		<SemanticGroup>Disorders</SemanticGroup>
	</UMLS>
</FocusAnnotations>
<QAPairs>
	<QAPair pid="1">
			<Question qid="0000001_1-1" qtype="information">What is (are) Adult Acute Lymphoblastic Leukemia ?</Question>
			<Answer>Key Points
                    - Adult acute lymphoblastic leukemia (ALL) is a type of cancer in which the bone marrow makes too many lymphocytes (a type of white blood cell).    - Leukemia may affect red blood cells, white blood cells, and platelets.    - Previous chemotherapy and exposure to radiation may increase the risk of developing ALL.    - Signs and symptoms of adult ALL include fever, feeling tired, and easy b

In [6]:
import os

# Recursively discover every XML file across all 12 source folders
# Configuration: Root directory for the dataset
root = "MedQuAD"
count = 0
fullpath = []       # stores the full path of every XML file found
all_dirnames = []   # stores each unique source folder encountered

# Recursively traverse the directory tree to find all XML files
for dirpath, dirnames, filenames in os.walk(root):
    for file in filenames:
        if file.endswith(".xml"):
            count += 1
            full_path = os.path.join(dirpath, file)
            fullpath.append(full_path)
            # Keep track of unique sub-directories (sources)
            if dirpath not in all_dirnames:
                all_dirnames.append(dirpath)

# Sanity check: should be 11,274 files across 12 directories
print(f"No. of XML files: {count}")
print(f"Directories found: {len(all_dirnames)}")

No. of XML files: 11274
Directories found: 12


In [7]:
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np

# Single, corrected parser: extracts every Question-Answer pair from every XML file
# Uses itertext() (not .text) so nested/formatted text inside tags is fully captured
data = []
error_count = 0
success_count = 0

# Loop through every XML file and extract QA pairs
for path in fullpath:
    try:
        tree = ET.parse(path)
        root_xml = tree.getroot()

        # Document-level metadata (same for every QAPair within this file)
        source = root_xml.get('source')
        document_id = root_xml.get('id')
        focus_elem = root_xml.find('Focus')
        focus = focus_elem.text if focus_elem is not None else ''

        # Each file can contain multiple Question-Answer pairs
        for qa_pair in root_xml.findall('.//QAPair'):
            question_elem = qa_pair.find('Question')
            answer_elem = qa_pair.find('Answer')

            # Capture text content and metadata (IDs and Types)
            # qtype/qid are attributes on the Question tag, not separate elements
            question = "".join(question_elem.itertext()).strip() if question_elem is not None else ''
            qtype = question_elem.get('qtype') if question_elem is not None else ''
            qid = question_elem.get('qid') if question_elem is not None else ''
            answer = "".join(answer_elem.itertext()).strip() if answer_elem is not None else None

            data.append({
                'doc_id': document_id,
                'focus': focus,
                'source': source,
                'qid': qid,
                'question': question,
                'answer': answer,
                'qtype': qtype,
            })
            success_count += 1

    except Exception as e:
      # Track and report any file that fails to parse
      error_count += 1
      print(f"Error parsing {path}: {e}")

# Create a DataFrame and check for empty/null answers
df = pd.DataFrame(data)

# Some Answer tags are genuinely empty in the source XML (see copyright note below)
# This flags those rows so they can be inspected and later excluded from training data
def is_empty(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return True
    return isinstance(val, str) and val.strip() == ''

df['is_missing_answer'] = df['answer'].apply(is_empty)
print(f"Extraction complete. Success: {success_count}, Missing Answers: {df['is_missing_answer'].sum()}")

Extraction complete. Success: 47441, Missing Answers: 31034


In [8]:
# Get the full list of unique qtypes and their frequency
qtype_unique = df['qtype'].unique()
qtype_unique_counts = df['qtype'].value_counts()

# Calculate the total sum of all qtype occurrences
total_qtype_sum = qtype_unique_counts.sum()

print("--- Count of each Question Type ---")
display(qtype_unique_counts)

# Sanity check: qtype counts should sum to the total number of successfully parsed QA pairs
print(f"\nTotal sum of all qtype occurrences: {total_qtype_sum}")
print(f"Verification: Matches success_count ({success_count})? {total_qtype_sum == success_count}")

--- Count of each Question Type ---


,count
qtype,
information,9214
symptoms,4338
treatment,3906
causes,2436
outlook,2232
exams and tests,2058
when to contact a medical professional,1738
inheritance,1446
precautions,1413



Total sum of all qtype occurrences: 47441
Verification: Matches success_count (47441)? True


In [9]:
import plotly.express as px

# Visualize the qtype distribution to spot skew at a glance
# (e.g., 'information' dominates while many qtypes are long-tail/rare)

# 1. Calculate counts and percentages
qtype_stats = df['qtype'].value_counts().reset_index()
qtype_stats.columns = ['qtype', 'count']
total_qa = qtype_stats['count'].sum()
qtype_stats['percentage'] = (qtype_stats['count'] / total_qa * 100).round(2)

# 2. Create interactive bar chart using Plotly
fig = px.bar(
    qtype_stats,
    x='count',
    y='qtype',
    orientation='h',
    title='Distribution of Question Types (MedQuAD)',
    labels={'count': 'Number of QA Pairs', 'qtype': 'Question Type'},
    # Pass percentage as custom data for the hover template
    custom_data=['percentage'],
    color='count',
    color_continuous_scale='Viridis',
    height=900
)

# 3. Customize hover template to show percentage
fig.update_traces(
    hovertemplate='<b>%{y}</b><br>Count: %{x}<br>Percentage: %{customdata[0]}%<extra></extra>'
)

fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.show()

### Sampling Ambiguous Question Types
Inspecting `research`, `when to contact a medical professional`, and `other information` to see if they lean towards diseases or drugs.

In [10]:
import pandas as pd

# Manually inspect qtypes whose disease-vs-drug category isn't obvious from the label alone
# List of qtypes to investigate
ambiguous_qtypes = ['research', 'when to contact a medical professional', 'other information']

for qt in ambiguous_qtypes:
    print(f"\n{'='*30} SAMPLES FOR: {qt} {'='*30}")
    # Filter the dataframe and take up to 4 samples
    samples = df[df['qtype'] == qt].head(4)

    for idx, row in samples.iterrows():
        print(f"\n[Focus]: {row['focus']}")
        print(f"[Question]: {row['question']}")
        print(f"[Answer Snippet]: {str(row['answer'])[:200]}...")
        print("-" * 20)


============================== SAMPLES FOR: research ==============================

[Focus]: Childhood Brain Stem Glioma
[Question]: what research (or clinical trials) is being done for Childhood Brain Stem Glioma ?
[Answer Snippet]: New types of treatment are being tested in clinical trials.
                 Information about clinical trials is available from the NCI website.
                
                
                    ...
--------------------

[Focus]: Endometrial Cancer
[Question]: what research (or clinical trials) is being done for Endometrial Cancer ?
[Answer Snippet]: New types of treatment are being tested in clinical trials.
                    Information about clinical trials is available from the NCI website.
                
                
                 ...
--------------------

[Focus]: Childhood Central Nervous System Germ Cell Tumors
[Question]: what research (or clinical trials) is being done for Childhood Central Nervous System Germ Cell Tumors ?
[Ans

In [11]:
# Break down missing-answer rate by qtype
# This is what revealed that certain qtypes (mostly drug-related) are 100% missing answers
missing_stats = df.groupby('qtype')['is_missing_answer'].agg(['sum', 'count']).reset_index()
missing_stats.columns = ['qtype', 'missing_count', 'total_count']
missing_stats['missing_percentage'] = (missing_stats['missing_count'] / missing_stats['total_count'] * 100).round(2)

missing_stats_sorted = missing_stats.sort_values(by='missing_count', ascending=False)

print("--- Missing Answers by Question Type ---")
display(missing_stats_sorted[missing_stats_sorted['missing_count'] > 0])

--- Missing Answers by Question Type ---


,qtype,missing_count,total_count,missing_percentage
18,information,4679,9214,50.78
24,outlook,1871,2232,83.83
37,when to contact a medical professional,1738,1738,100.00
2,causes,1709,2436,70.16
34,symptoms,1590,4338,36.65
35,treatment,1464,3906,37.48
25,precautions,1413,1413,100.00
9,exams and tests,1405,2058,68.27
29,side effects,1301,1301,100.00
23,other information,1280,1280,100.00


In [12]:
# Manually verify one missing-answer row against the raw XML to confirm this is a
# genuine source-data gap, not a parser bug

# 1. Identify the specific row with both focus and qtype
target_row = df[(df['focus'] == 'Brucellosis') & (df['qtype'] == 'when to contact a medical professional')].iloc[0]
target_id = target_row['doc_id']
target_source = target_row['source']

# 2. Narrow down the path search using both source folder AND doc_id
# (doc_id alone is not unique across folders, so both are required for an exact match)
target_file_path = [p for p in fullpath if target_source in p and target_id in p][0]

print(f"Target ID: {target_id}")
print(f"Target Source: {target_source}")
print(f"Target File: {target_file_path}")
print(f"Parsed Question: {target_row['question']}")
print(f"Parsed Answer (Current): '{target_row['answer']}'")

# 3. Output the raw XML content to inspect the <Answer> tag directly
print("\n--- RAW XML CONTENT ---")
with open(target_file_path, 'r') as f:
    print(f.read())

Target ID: 0000598
Target Source: ADAM
Target File: MedQuAD/10_MPlus_ADAM_QA/0000598.xml
Parsed Question: Do I need to see a doctor for Brucellosis ?
Parsed Answer (Current): ''

--- RAW XML CONTENT ---
<?xml version="1.0" encoding="UTF-8"?>
<Document id="0000598" source="ADAM" url="https://www.nlm.nih.gov/medlineplus/ency/article/000597.htm">
 <!--Answers from the A.D.A.M. medical encyclopedia were removed to comply with the MedlinePlus copyright-->
<Focus>Brucellosis</Focus>
<FocusAnnotations>
	<Category>Disease</Category>
	<UMLS>
		<CUIs>
			<CUI>C0006309</CUI>
		</CUIs>
		<SemanticTypes>
			<SemanticType>T047</SemanticType>
		</SemanticTypes>
		<SemanticGroup>Disorders</SemanticGroup>
	</UMLS>
	<Synonyms>
		<Synonym>Cyprus fever</Synonym>
		<Synonym>Undulant fever</Synonym>
		<Synonym>Gibraltar fever</Synonym>
		<Synonym>Malta fever</Synonym>
		<Synonym>Mediterranean fever</Synonym>
	</Synonyms>
</FocusAnnotations>
<QAPairs>
	<QAPair pid="1">
			<Question qid="0000598-1" qtype="inf

In [13]:
# Build the final usable dataset by dropping rows with no ground-truth answer
# (These are unusable for SFT since there's no target to train against)

# 1. Filter out records where answers were removed due to copyright
df_final = df[~df['is_missing_answer']].copy()
df_final.drop(columns=['is_missing_answer'], inplace=True)

# 2. Display summary of the usable dataset
print(f"Final Usable Row Count: {len(df_final)}")

# 3. Export to a local CSV file
output_path = os.path.abspath('medquad_cleaned.csv')
df_final.to_csv(output_path, index=False)
print(f"Local export successful: {output_path}")

Final Usable Row Count: 16407
Local export successful: /content/medquad_cleaned.csv


In [15]:
# Mount Google Drive so the cleaned dataset persists across Colab sessions
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
# Copy the cleaned CSV from local Colab storage into the project's Drive folder
# 1. Copy the cleaned file
!cp /content/medquad_cleaned.csv /content/drive/MyDrive/smollm2-medical-sft/data/processed/

print("File successfully saved to Google Drive!")


File successfully saved to Google Drive!


In [17]:
# Verify the file landed correctly in the Drive folder
!ls /content/drive/MyDrive/smollm2-medical-sft/data/processed/

medquad_cleaned.csv
